In [1]:
import numpy as np

def density_matrix(state):
    return np.outer(state, state.conj())


def measurement_probabilities(state):
    return np.abs(state) ** 2


def purity(rho):
    return np.real(np.trace(rho @ rho))


def check_density_matrix(rho):
    hermitian = np.allclose(rho, rho.conj().T)
    trace_one = np.isclose(np.trace(rho), 1)
    eigenvalues = np.linalg.eigvalsh(rho)
    positive = np.all(eigenvalues >= -1e-10)

    return hermitian and trace_one and positive


def partial_trace(rho, keep):
    rho_4d = rho.reshape(2, 2, 2, 2)

    if keep == 0:
        return np.einsum('abcb->ac', rho_4d)
    elif keep == 1:
        return np.einsum('abad->bd', rho_4d)
    else:
        raise ValueError("keep must be 0 or 1")


def analyze_pure_two_qubit_state(state):
    rho = density_matrix(state)

    rho_A = partial_trace(rho, keep=0)
    rho_B = partial_trace(rho, keep=1)

    return {
        "joint_density_matrix": rho,
        "joint_purity": purity(rho),
        "reduced_A": rho_A,
        "reduced_B": rho_B,
        "purity_A": purity(rho_A),
        "purity_B": purity(rho_B),
        "entangled": (
            purity(rho_A) < 1 - 1e-10
        )
    }

Exercise 1 — Measurement and the Born rule

Tasks:
1. Represent the state as a NumPy complex array.
2. Calculate its measurement probabilities.
3. Confirm that the probabilities sum to one.
4. Simulate 10,000 measurements.
5. Display the measurement counts and empirical probabilities.

In [10]:
state = np.array([
    np.sqrt(2) / np.sqrt(3),
    1 / np.sqrt(3)
], dtype=complex)

probabilities = measurement_probabilities(state)

print("Probabilities:", probabilities)
print("Total:", np.sum(probabilities))

def simulate_measurements(state, shots=10000):
    probabilities = measurement_probabilities(state)

    outcomes = np.random.choice(
        len(state),
        size=shots,
        p=probabilities
    )

    return np.bincount(outcomes, minlength=len(state))

counts = simulate_measurements(state)

print("Counts:", counts)
print("Empirical Probabilities:", counts / 10000)   

Probabilities: [0.66666667 0.33333333]
Total: 1.0000000000000002
Counts: [6643 3357]
Empirical Probabilities: [0.6643 0.3357]


Exercise 2 — Measurement in different bases

Tasks:
1. Construct the Hadamard matrix.
2. Calculate computational-basis measurement probabilities.
3. Apply the Hadamard gate to the state.
4. Calculate the resulting measurement probabilities.
5. Simulate 10,000 measurements before and after applying the Hadamard gate.

In [12]:
H = (1 / np.sqrt(2)) * np.array([
    [1, 1],
    [1, -1]
], dtype=complex)

plus = np.array([
    1 / np.sqrt(2),
    1 / np.sqrt(2)
], dtype=complex)

print("Probabilities for |+⟩:", measurement_probabilities(plus))

# Transform into the computational basis
transformed_state = H @ plus

print(transformed_state)
print("Probabilities for H|+⟩:", measurement_probabilities(transformed_state))

Probabilities for |+⟩: [0.5 0.5]
[1.+0.j 0.+0.j]
Probabilities for H|+⟩: [1. 0.]


Exercise 3 — Density matrix construction

Tasks:
1. Construct their density matrices.
2. Print all three matrices.
3. Check that each matrix is Hermitian.
4. Check that each has trace one.
5. Calculate and display their eigenvalues.

In [16]:
zero = np.array([1, 0], dtype=complex)

one = np.array([0, 1], dtype=complex)

def density_matrix(state):
    return np.outer(state, state.conj())

rho_zero = density_matrix(zero)
print("Density matrix for |0⟩:\n", rho_zero)

rho_one = density_matrix(one)
print("Density matrix for |1⟩:\n", rho_one)

rho_plus = density_matrix(plus)
print("Density matrix for |+⟩:\n", rho_plus)

def check_density_matrix(rho):
    hermitian = np.allclose(rho, rho.conj().T)
    trace_one = np.isclose(np.trace(rho), 1)

    eigenvalues = np.linalg.eigvalsh(rho)
    positive = np.all(eigenvalues >= -1e-10)

    return {
        "Hermitian": hermitian,
        "Trace equals one": trace_one,
        "Positive semidefinite": positive
    }

print("Validation for rho_zero:", check_density_matrix(rho_zero))
print("Validation for rho_one:", check_density_matrix(rho_one))
print("Validation for rho_plus:", check_density_matrix(rho_plus))

eigenvalues_rho_zero = np.linalg.eigvalsh(rho_zero)
print("Eigenvalues of rho_zero:", eigenvalues_rho_zero)

eigenvalues_rho_one = np.linalg.eigvalsh(rho_one)
print("Eigenvalues of rho_one:", eigenvalues_rho_one)

eigenvalues_rho_plus = np.linalg.eigvalsh(rho_plus)
print("Eigenvalues of rho_plus:", eigenvalues_rho_plus)

Density matrix for |0⟩:
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
Density matrix for |1⟩:
 [[0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]
Density matrix for |+⟩:
 [[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]
Validation for rho_zero: {'Hermitian': True, 'Trace equals one': np.True_, 'Positive semidefinite': np.True_}
Validation for rho_one: {'Hermitian': True, 'Trace equals one': np.True_, 'Positive semidefinite': np.True_}
Validation for rho_plus: {'Hermitian': True, 'Trace equals one': np.True_, 'Positive semidefinite': np.True_}
Eigenvalues of rho_zero: [0. 1.]
Eigenvalues of rho_one: [0. 1.]
Eigenvalues of rho_plus: [0. 1.]


Exercise 4 — Pure versus mixed states

Tasks:
1. Represent both matrices using NumPy.
2. Calculate their purity.
3. Calculate their computational-basis measurement probabilities.
4. Apply a Hadamard transformation to both density matrices:
5. Calculate their measurement probabilities after the transformation.

In [22]:
rho_pure = 0.5 * np.array([
    [1, 1],
    [1, 1]
], dtype=complex)

rho_mixed = 0.5 * np.array([
    [1, 0],
    [0, 1]
], dtype=complex)

print("Purity of rho_pure:", purity(rho_pure))
print("Purity of rho_mixed:", purity(rho_mixed))

print("Measurement probabilities for rho_pure:", measurement_probabilities(rho_pure))
print("Measurement probabilities for rho_mixed:", measurement_probabilities(rho_mixed))

H_rho_pure = H @ (rho_pure @ (H.conj().T))
H_rho_mixed = H @ (rho_mixed @ (H.conj().T))
print("Transformed density matrix for rho_pure:\n", H_rho_pure)
print("Transformed density matrix for rho_mixed:\n", H_rho_mixed)

print("Measurement probabilities for H_rho_pure:\n", measurement_probabilities(H_rho_pure))
print("Measurement probabilities for H_rho_mixed:\n", measurement_probabilities(H_rho_mixed))

Purity of rho_pure: 1.0
Purity of rho_mixed: 0.5
Measurement probabilities for rho_pure: [[0.25 0.25]
 [0.25 0.25]]
Measurement probabilities for rho_mixed: [[0.25 0.  ]
 [0.   0.25]]
Transformed density matrix for rho_pure:
 [[ 1.00000000e+00+0.j  0.00000000e+00+0.j]
 [-2.23711432e-17+0.j  0.00000000e+00+0.j]]
Transformed density matrix for rho_mixed:
 [[ 5.00000000e-01+0.j -1.11855716e-17+0.j]
 [-1.11855716e-17+0.j  5.00000000e-01+0.j]]
Measurement probabilities for H_rho_pure:
 [[1.00000000e+00 0.00000000e+00]
 [5.00468047e-34 0.00000000e+00]]
Measurement probabilities for H_rho_mixed:
 [[2.50000000e-01 1.25117012e-34]
 [1.25117012e-34 2.50000000e-01]]


Exercise 5 — Partial trace

Construct the phi plus Bell state

Tasks:
1. Construct its state vector.
2. Calculate its joint density matrix.
3. Implement a reusable partial trace function.
4. Calculate the reduced density matrix of qubit A.
5. Calculate the reduced density matrix of qubit B.
6. Calculate the purity of both reduced states.

In [24]:
def combine_qubits(qubit_a, qubit_b):
    return np.kron(qubit_a, qubit_b)

def partial_trace(rho, keep):
    """
    Partial trace of a two-qubit density matrix.

    keep=0: retain first qubit, trace out second.
    keep=1: retain second qubit, trace out first.
    """

    rho_4d = rho.reshape(2, 2, 2, 2)

    if keep == 0:
        return np.einsum('abcb->ac', rho_4d)

    elif keep == 1:
        return np.einsum('abad->bd', rho_4d)

    else:
        raise ValueError("keep must be 0 or 1")


CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

I = np.eye(2, dtype=complex)

state_00 = combine_qubits(zero, zero)
H_first_gate = np.kron(H, I)
H_on_first = H_first_gate @ state_00
bell_state = CNOT @ H_on_first
print("Bell state |Φ+⟩:\n", bell_state)

density_bell = density_matrix(bell_state)
print("Density matrix of Bell state:\n", density_bell)

rho_A = partial_trace(density_bell, keep=0)
rho_B = partial_trace(density_bell, keep=1)

print("Reduced density matrix for qubit A:\n", rho_A)
print("Reduced density matrix for qubit B:\n", rho_B)

print("Purity of reduced density matrix A:", purity(rho_A))
print("Purity of reduced density matrix B:", purity(rho_B))

Bell state |Φ+⟩:
 [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Density matrix of Bell state:
 [[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]
Reduced density matrix for qubit A:
 [[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Reduced density matrix for qubit B:
 [[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Purity of reduced density matrix A: 0.4999999999999998
Purity of reduced density matrix B: 0.4999999999999998


Exercise 6 — Entanglement analysis

Investigate three states:

Tasks:
1. Construct each state vector.
2. Generate its joint density matrix.
3. Calculate both reduced density matrices.
4. Calculate the purity of each reduced state.
5. Classify each state as separable or entangled using the pure-state reduced-purity test.

In [25]:
state_1 = combine_qubits(zero, zero)
state_2 = (1 / np.sqrt(2)) * state_1 + (1 / np.sqrt(2)) * combine_qubits(one, one)
state_3 = (1 / np.sqrt(2)) * combine_qubits(zero, one) + (1 / np.sqrt(2)) * combine_qubits(one, zero)

density_bell = density_matrix(state_2)
density_bell_2 = density_matrix(state_3)

rho_A = partial_trace(density_bell, keep=0)
rho_B = partial_trace(density_bell, keep=1)

rho_2A = partial_trace(density_bell_2, keep=0)
rho_2B = partial_trace(density_bell_2, keep=1)

print("Purity of reduced density matrix A (Bell state):", purity(rho_A))
print("Purity of reduced density matrix B (Bell state):", purity(rho_B))

print("Purity of reduced density matrix A (Bell state 2):", purity(rho_2A))
print("Purity of reduced density matrix B (Bell state 2):", purity(rho_2B))

print("Is the first Bell state entangled?", purity(rho_A) < 1 - 1e-10)
print("Is the second Bell state entangled?", purity(rho_2A) < 1 - 1e-10)

Purity of reduced density matrix A (Bell state): 0.4999999999999998
Purity of reduced density matrix B (Bell state): 0.4999999999999998
Purity of reduced density matrix A (Bell state 2): 0.4999999999999998
Purity of reduced density matrix B (Bell state 2): 0.4999999999999998
Is the first Bell state entangled? True
Is the second Bell state entangled? True


Exercise 7 — Build a reusable state analyzer

Using the functions developed throughout today's session, build one reusable function that accepts a normalized two-qubit state vector and returns:

- Joint density matrix.
- Joint purity.
- Reduced density matrix of qubit A.
- Reduced density matrix of qubit B.
- Purity of qubit A.
- Purity of qubit B.
- Entanglement classification.

In [26]:
def analyze_pure_two_qubit_state(state):
    rho = density_matrix(state)

    rho_A = partial_trace(rho, keep=0)
    rho_B = partial_trace(rho, keep=1)

    return {
        "joint_density_matrix": rho,
        "joint_purity": purity(rho),
        "reduced_A": rho_A,
        "reduced_B": rho_B,
        "purity_A": purity(rho_A),
        "purity_B": purity(rho_B),
        "entangled": (
            purity(rho_A) < 1 - 1e-10
        )
    }

print("Analysis of first Bell state:", analyze_pure_two_qubit_state(state_2))
print("Analysis of second Bell state:", analyze_pure_two_qubit_state(state_3))

Analysis of first Bell state: {'joint_density_matrix': array([[0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j]]), 'joint_purity': np.float64(0.9999999999999996), 'reduced_A': array([[0.5+0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j]]), 'reduced_B': array([[0.5+0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j]]), 'purity_A': np.float64(0.4999999999999998), 'purity_B': np.float64(0.4999999999999998), 'entangled': np.True_}
Analysis of second Bell state: {'joint_density_matrix': array([[0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j, 0.5+0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j, 0.5+0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j]]), 'joint_purity': np.float64(0.9999999999999996), 'reduced_A': array([[0.5+0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j]]), 'reduced_B': array([[0.5+0.j, 0. +0.j],
       [0. +0.j, 0.5+0.j]]), 'purity_A': np.float64(0.4999999999999998)